# Data assimilation: a field from scattered sensors

**Book:** §4.5, Figure 4.3(a,b) &nbsp;·&nbsp; `ch04/data_assimilation.ipynb`

Steady conduction on the unit square, $\nabla^2 T=0$. We are given **40 noisy point sensors** in
the interior and **no boundary conditions at all** — a problem no finite-difference or
finite-element code can even be asked, because they require a closed boundary.

$$\mathcal L=\underbrace{\overline{(\nabla^2 T)^2}}_{\text{physics everywhere}}
\;+\;50\,\underbrace{\frac1{N_d}\sum_{i=1}^{40}\big(T(x_i,y_i)-T_i^{\rm obs}\big)^2}_{\text{the 40 sensors}}$$

The physics fills in everywhere the sensors are silent — *including the boundary*, which comes out
as a by-product. Truth: $T=\sin\pi x\,\sinh\pi y/\sinh\pi$ (used only to make the data and to
score).

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
np.random.seed(0); torch.manual_seed(0)
def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]
def mlp(s):
    L = []
    for i in range(len(s)-1):
        L.append(nn.Linear(s[i], s[i+1]))
        if i < len(s)-2: L.append(nn.Tanh())
    return nn.Sequential(*L)
rel = lambda p, e: float(np.sqrt(np.mean((p-e)**2)/np.mean(e**2)))
PI = np.pi
Tex = lambda x, y: np.sin(PI*x)*np.sinh(PI*y)/np.sinh(PI)

NS = 40                                     # forty sensors, scattered at random
xs, ys = np.random.rand(NS), np.random.rand(NS)
ts = Tex(xs, ys) + 0.01*np.random.randn(NS)
xs_t = torch.tensor(xs, dtype=torch.float32).reshape(-1,1)
ys_t = torch.tensor(ys, dtype=torch.float32).reshape(-1,1)
ts_t = torch.tensor(ts, dtype=torch.float32).reshape(-1,1)

net = mlp([2,64,64,64,1]); opt = torch.optim.Adam(net.parameters(), 2e-3)
t0 = time.perf_counter()
for e in range(15000):
    if e ==  9000:
        for g in opt.param_groups: g['lr'] = 5e-4
    if e == 12750:
        for g in opt.param_groups: g['lr'] = 1e-4
    opt.zero_grad()
    x = torch.rand(3000,1).requires_grad_(True)
    y = torch.rand(3000,1).requires_grad_(True)
    T = net(torch.cat([x,y],1))
    lap = g1(g1(T,x),x) + g1(g1(T,y),y)                    # physics, everywhere
    data = ((net(torch.cat([xs_t,ys_t],1)) - ts_t)**2).mean()   # 40 sensors
    ((lap**2).mean() + 50*data).backward(); opt.step()      # NO boundary conditions
print(f'training: {time.perf_counter()-t0:.0f} s')

n = 101; gx = np.linspace(0,1,n); X, Y = np.meshgrid(gx, gx, indexing='ij')
P = torch.tensor(np.stack([X.ravel(), Y.ravel()],1), dtype=torch.float32)
with torch.no_grad(): Tp = net(P).numpy().reshape(n,n)
Te = Tex(X, Y)
eA = rel(Tp, Te); eB = rel(Tp[:,-1], Te[:,-1])
print(f'field   rel L2 = {eA:.3f}')
print(f'top BC  rel L2 = {eB:.3f}   <-- never supplied!')

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.3))
c0 = ax[0].contourf(X, Y, Tp, 21, cmap='inferno'); plt.colorbar(c0, ax=ax[0])
ax[0].scatter(xs, ys, c='cyan', s=22, edgecolors='k', linewidths=.5, label=f'{NS} sensors')
ax[0].set_title(f'(a) Field from sensors + physics alone (rel $L_2$={eA:.3f})', fontsize=11)
ax[0].set_xlabel('x'); ax[0].set_ylabel('y'); ax[0].legend(fontsize=8, loc='lower left')
ax[1].plot(gx, Te[:,-1], 'g', lw=2.8, alpha=.6, label='true boundary value')
ax[1].plot(gx, Tp[:,-1], 'r--', lw=1.7, label=f'PINN, inferred (rel $L_2$={eB:.3f})')
ax[1].set_title('(b) The boundary condition, never given, is recovered', fontsize=11)
ax[1].set_xlabel('x'); ax[1].set_ylabel('T(x, y=1)'); ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()